In [21]:
import pandas as pd
import numpy as np
from scipy import stats
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

In [22]:
df_data = pd.read_excel("/Volumes/T7 1TB SSD/Empirical_Analysis_FM/Applied_Research_Framework_Git/Applied_projects/Crude_oil_market_analysis/Regression_Urals_loading_analysis_mod.xlsx", sheet_name="Data")
print(df_data)

          Date  LCOc1  NWEMURLCRKMc1   .IMOEX    RSX  BFO-URL-NWE  \
0   2017-01-06  56.75      -0.545276  2213.93  21.52        -2.50   
1   2017-01-13  55.59       1.238238  2195.19  21.45        -2.05   
2   2017-01-20  55.49       0.386498  2159.96  21.08        -1.50   
3   2017-01-27  55.46       0.348593  2266.05  22.08        -1.60   
4   2017-02-03  56.72       0.219638  2226.61  21.80        -1.75   
..         ...    ...            ...      ...    ...          ...   
230 2021-10-22  85.77       7.090000  4196.96  32.67        -1.80   
231 2021-10-29  84.38       3.930000  4150.00  32.00        -1.40   
232 2021-11-05  82.55       1.860000  4174.76  31.93        -1.60   
233 2021-11-12  81.95       1.750000  4121.66  31.27        -1.45   
234 2021-11-19  78.66       0.030000  4016.47  29.92        -1.65   

     Urals loading  
0     1.391864e+07  
1     1.480585e+07  
2     1.377569e+07  
3     1.332605e+07  
4     1.538791e+07  
..             ...  
230   1.347257e+07  
231

In [23]:
# complete correlation structure analysis including Urals loading
merged_df = df_data[['Urals loading', 'NWEMURLCRKMc1', '.IMOEX','LCOc1']]

print(merged_df)

df_data_standardized = (merged_df - merged_df.mean()) / merged_df.std()
df_data_zscore = df_data_standardized[(np.abs(stats.zscore(df_data_standardized)) < 3).all(axis=1)]

corr_matrix_merged = df_data_zscore[['Urals loading', 'NWEMURLCRKMc1', '.IMOEX','LCOc1']].corr()

     Urals loading  NWEMURLCRKMc1   .IMOEX  LCOc1
0     1.391864e+07      -0.545276  2213.93  56.75
1     1.480585e+07       1.238238  2195.19  55.59
2     1.377569e+07       0.386498  2159.96  55.49
3     1.332605e+07       0.348593  2266.05  55.46
4     1.538791e+07       0.219638  2226.61  56.72
..             ...            ...      ...    ...
230   1.347257e+07       7.090000  4196.96  85.77
231   1.275017e+07       3.930000  4150.00  84.38
232   9.529337e+06       1.860000  4174.76  82.55
233   3.400765e+06       1.750000  4121.66  81.95
234   1.018841e+07       0.030000  4016.47  78.66

[235 rows x 4 columns]


In [24]:
corr_4x4_actual = np.array([
    [1.00,    -0.024,  -0.409,  0.229],    # Urals loading 
    [-0.024,  1.00,     0.219,  0.221],    # NWEMURLCRKMc1   
    [-0.409,  0.219,    1.00,   0.292],    # .IMOEX    
    [0.229,   0.221,    0.292,  1.00]      # LCOc1
    ])

In [25]:

means = df_data[['Urals loading', 'NWEMURLCRKMc1', '.IMOEX', 'LCOc1']].mean().values
stds = df_data[['Urals loading', 'NWEMURLCRKMc1', '.IMOEX', 'LCOc1']].std().values

In [26]:
cov_4x4 = np.outer(stds, stds) * corr_4x4_actual

In [27]:
# generate synthetic
n_obs = 234
np.random.seed(42)
synthetic_data = np.random.multivariate_normal(
    mean=means,
    cov=cov_4x4,
    size=234)

print(f"✓ generated successfully: shape {synthetic_data.shape}")

✓ generated successfully: shape (234, 4)


In [28]:
# verification synthetic has same correlations as real

df_synthetic = pd.DataFrame(synthetic_data,
columns=['Urals loading', 'NWEMURL.CRMc1', '.IMOEX', 'LCOc1'])
print("real correlations:")
print(corr_matrix_merged)

print("\nsynthetic correlations:")
print(df_synthetic.corr())

real correlations:
               Urals loading  NWEMURLCRKMc1    .IMOEX     LCOc1
Urals loading       1.000000      -0.023755 -0.409401  0.229433
NWEMURLCRKMc1      -0.023755       1.000000  0.218897  0.221110
.IMOEX             -0.409401       0.218897  1.000000  0.291510
LCOc1               0.229433       0.221110  0.291510  1.000000

synthetic correlations:
               Urals loading  NWEMURL.CRMc1    .IMOEX     LCOc1
Urals loading       1.000000       0.023830 -0.423747  0.227258
NWEMURL.CRMc1       0.023830       1.000000  0.243727  0.244638
.IMOEX             -0.423747       0.243727  1.000000  0.228545
LCOc1               0.227258       0.244638  0.228545  1.000000


In [29]:
start_date = pd.Timestamp('2017-01-06')
df_synthetic['date'] = pd.date_range(start=start_date, periods=n_obs, freq='W')

# reorder columns: date first, then data
df_synthetic = df_synthetic[['date','Urals loading', 'NWEMURL.CRMc1', '.IMOEX', 'LCOc1']]

In [30]:
df_synthetic.to_csv('urals_synthetic_data.csv', index=False)

print(f"\nfirst 10 rows of synthetic data:")
print(df_synthetic.head(10))
print(f"\ndata shape: {df_synthetic.shape}")
print(f"columns: {list(df_synthetic.columns)}")


first 10 rows of synthetic data:
        date  Urals loading  NWEMURL.CRMc1       .IMOEX      LCOc1
0 2017-01-08   1.059703e+07      -2.437084  2919.842865  67.715078
1 2017-01-15   1.284399e+07       0.482367  2789.741086  80.438358
2 2017-01-22   1.356745e+07       2.705914  2294.991972  54.363780
3 2017-01-29   1.138023e+07       4.263067  3851.931309  51.458729
4 2017-02-05   1.523793e+07       5.612465  2286.495648  52.100909
5 2017-02-12   7.618168e+06       6.782592  3212.656670  58.566130
6 2017-02-19   1.379775e+07      -0.019098  2518.407980  49.441688
7 2017-02-26   1.397070e+07      -4.125038  2730.104236  58.000781
8 2017-03-05   1.216561e+07       7.052637  3307.386982  75.546454
9 2017-03-12   1.148199e+07       2.090782  3869.596600  56.276376

data shape: (234, 5)
columns: ['date', 'Urals loading', 'NWEMURL.CRMc1', '.IMOEX', 'LCOc1']


In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════════════════════════════
Synthetic Data Validation Suite v2.0
Oil commodities Desk - Urals Loading ML Forecasting

Statistical Tests for JPM Alternative Data Conference
Author: GS Quant Desk
Date: November 2025
═══════════════════════════════════════════════════════════════════════════════════════════════════════

OVERVIEW:
---------
This module implements a comprehensive statistical validation framework for comparing
synthetic data against real observable data. Essential for validating ML training data
in commodity forecasting applications.

TESTS IMPLEMENTED:
------------------
1. Descriptive Statistics (mean, std, skewness, kurtosis, percentiles)
2. Kolmogorov-Smirnov Two-Sample Test (distribution comparison)
3. Anderson-Darling Test (tail-sensitive normality)
4. Jarque-Bera Test (combined skew/kurtosis normality)
5. D'Agostino-Pearson Test (normality)
6. Mann-Whitney U Test (non-parametric comparison)
7. Levene's Test (variance equality)
8. Correlation Structure Validation
9. Tail Risk Analysis (VaR percentiles)

USAGE:
------
    python synthetic_validation_suite.py --real data_real.csv --synthetic data_synth.csv

"""

"\n═══════════════════════════════════════════════════════════════════════════════════════════════════════\nGOLDMAN SACHS QUANTITATIVE STRATEGIES\nSynthetic Data Validation Suite v2.0\nOil Commodities Desk - Urals Loading ML Forecasting\n\nStatistical Tests for JPM Alternative Data Conference\nAuthor: GS Quant Desk\nDate: November 2025\n═══════════════════════════════════════════════════════════════════════════════════════════════════════\n\nOVERVIEW:\n---------\nThis module implements a comprehensive statistical validation framework for comparing\nsynthetic data against real observable data. Essential for validating ML training data\nin commodity forecasting applications.\n\nTESTS IMPLEMENTED:\n------------------\n1. Descriptive Statistics (mean, std, skewness, kurtosis, percentiles)\n2. Kolmogorov-Smirnov Two-Sample Test (distribution comparison)\n3. Anderson-Darling Test (tail-sensitive normality)\n4. Jarque-Bera Test (combined skew/kurtosis normality)\n5. D'Agostino-Pearson Test (n

In [32]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

class Config:
    """Configuration for statistical tests"""
    ALPHA = 0.05  # Significance level
    KS_THRESHOLD = 0.05
    AD_CRITICAL = 0.787  # Anderson-Darling critical value at 5%
    CORR_ERROR_EXCELLENT = 0.03
    CORR_ERROR_GOOD = 0.05
    CORR_ERROR_ACCEPTABLE = 0.10

In [33]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 1: DESCRIPTIVE STATISTICS
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def compute_descriptive_stats(series: pd.Series, name: str) -> Dict:
    """
    Compute comprehensive descriptive statistics for a series.
    
    Parameters:
    -----------
    series : pd.Series
        Data series to analyze
    name : str
        Variable name for labeling
        
    Returns:
    --------
    dict : Dictionary containing all descriptive statistics
    """
    return {
        'Variable': name,
        'N': len(series),
        'Mean': series.mean(),
        'Std': series.std(),
        'Min': series.min(),
        'Q1': series.quantile(0.25),
        'Median': series.median(),
        'Q3': series.quantile(0.75),
        'Max': series.max(),
        'Range': series.max() - series.min(),
        'IQR': series.quantile(0.75) - series.quantile(0.25),
        'CV_pct': (series.std() / series.mean() * 100) if series.mean() != 0 else np.nan,
        'Skewness': stats.skew(series),
        'Kurtosis': stats.kurtosis(series),  # Excess kurtosis
        'P1': series.quantile(0.01),
        'P5': series.quantile(0.05),
        'P95': series.quantile(0.95),
        'P99': series.quantile(0.99)
    }


def compare_descriptive_stats(real: pd.Series, synthetic: pd.Series, var_name: str) -> Dict:
    """
    Compare descriptive statistics between real and synthetic data.
    
    Returns percentage differences for key metrics.
    """
    real_stats = compute_descriptive_stats(real, f"Real_{var_name}")
    synth_stats = compute_descriptive_stats(synthetic, f"Synth_{var_name}")
    
    mean_diff_pct = abs(real_stats['Mean'] - synth_stats['Mean']) / abs(real_stats['Mean']) * 100
    std_diff_pct = abs(real_stats['Std'] - synth_stats['Std']) / abs(real_stats['Std']) * 100
    
    return {
        'Variable': var_name,
        'Real_Mean': real_stats['Mean'],
        'Synth_Mean': synth_stats['Mean'],
        'Mean_Error_Pct': mean_diff_pct,
        'Real_Std': real_stats['Std'],
        'Synth_Std': synth_stats['Std'],
        'Std_Error_Pct': std_diff_pct,
        'Real_Skewness': real_stats['Skewness'],
        'Synth_Skewness': synth_stats['Skewness'],
        'Real_Kurtosis': real_stats['Kurtosis'],
        'Synth_Kurtosis': synth_stats['Kurtosis']
    }

In [34]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 2: DISTRIBUTION SHAPE ANALYSIS (SKEWNESS & KURTOSIS)
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def analyze_distribution_shape(series: pd.Series) -> Dict:
    """
    Analyze distribution shape through skewness and kurtosis.
    
    Skewness interpretation:
        |skew| < 0.5: Approximately symmetric
        |skew| < 1.0: Moderate skew
        |skew| > 1.0: Highly skewed
    
    Kurtosis interpretation (excess):
        kurt ≈ 0: Normal tails (mesokurtic)
        kurt > 0: Fat tails (leptokurtic) - more extreme events
        kurt < 0: Thin tails (platykurtic) - fewer extreme events
    """
    skewness = stats.skew(series)
    kurtosis = stats.kurtosis(series)
    
    # Interpret skewness
    if abs(skewness) < 0.5:
        skew_interp = "Symmetric"
    elif abs(skewness) < 1.0:
        skew_interp = "Moderate skew"
    else:
        skew_interp = "Highly skewed"
    
    # Interpret kurtosis
    if abs(kurtosis) < 1:
        kurt_interp = "Normal tails"
    elif kurtosis > 1:
        kurt_interp = "Fat tails (risk)"
    else:
        kurt_interp = "Thin tails"
    
    return {
        'Skewness': skewness,
        'Skewness_Interpretation': skew_interp,
        'Kurtosis': kurtosis,
        'Kurtosis_Interpretation': kurt_interp
    }

In [35]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 3: KOLMOGOROV-SMIRNOV TWO-SAMPLE TEST
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def ks_two_sample_test(real: pd.Series, synthetic: pd.Series) -> Dict:
    """
    Kolmogorov-Smirnov two-sample test for distribution comparison.
    
    H0: Both samples come from the same distribution
    H1: Samples come from different distributions
    
    The KS statistic measures the maximum distance between empirical CDFs.
    
    Parameters:
    -----------
    real : pd.Series
        Real data series
    synthetic : pd.Series
        Synthetic data series
        
    Returns:
    --------
    dict : Test results including statistic, p-value, and interpretation
    """
    ks_stat, p_value = stats.ks_2samp(real.dropna(), synthetic.dropna())
    
    passed = p_value > Config.ALPHA
    
    if p_value > 0.10:
        quality = "EXCELLENT"
    elif p_value > 0.05:
        quality = "GOOD"
    else:
        quality = "POOR"
    
    return {
        'Test': 'Kolmogorov-Smirnov',
        'Statistic': ks_stat,
        'p_value': p_value,
        'Alpha': Config.ALPHA,
        'Passed': passed,
        'Quality': quality,
        'Interpretation': f"Distributions are {'statistically similar' if passed else 'significantly different'}"
    }


In [36]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 4: ANDERSON-DARLING TEST (TAIL-SENSITIVE)
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def anderson_darling_test(series: pd.Series) -> Dict:
    """
    Anderson-Darling test for normality.
    
    More sensitive to distribution tails than KS test.
    Critical for VaR/CVaR risk modeling where tails matter.
    
    H0: Data follows normal distribution
    H1: Data does not follow normal distribution
    """
    # Standardize the data
    standardized = (series - series.mean()) / series.std()
    
    result = stats.anderson(standardized, dist='norm')
    
    # Critical value at 5% significance (index 2)
    critical_5pct = result.critical_values[2]
    passed = result.statistic < critical_5pct
    
    return {
        'Test': 'Anderson-Darling',
        'Statistic': result.statistic,
        'Critical_Value_5pct': critical_5pct,
        'Passed': passed,
        'Interpretation': f"Data is {'approximately normal' if passed else 'non-normal'}"
    }

In [37]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 5: JARQUE-BERA TEST
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def jarque_bera_test(series: pd.Series) -> Dict:
    """
    Jarque-Bera test for normality.
    
    Combines skewness and kurtosis into a single test statistic.
    JB = (n/6) * [S² + (K²/4)]
    
    Widely used in econometrics and finance.
    
    H0: Data follows normal distribution (S=0, K=0)
    H1: Data does not follow normal distribution
    """
    jb_stat, p_value = stats.jarque_bera(series.dropna())
    
    passed = p_value > Config.ALPHA
    
    return {
        'Test': 'Jarque-Bera',
        'Statistic': jb_stat,
        'p_value': p_value,
        'Passed': passed,
        'Interpretation': f"Data is {'approximately normal' if passed else 'non-normal'}"
    }

In [38]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 6: D'AGOSTINO-PEARSON TEST
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def dagostino_pearson_test(series: pd.Series) -> Dict:
    """
    D'Agostino-Pearson test for normality.
    
    Tests whether skewness and kurtosis significantly deviate from normal.
    Combines tests for skewness and kurtosis.
    
    H0: Data follows normal distribution
    H1: Data does not follow normal distribution
    """
    try:
        stat, p_value = stats.normaltest(series.dropna())
        passed = p_value > Config.ALPHA
        
        return {
            'Test': 'DAgostino-Pearson',
            'Statistic': stat,
            'p_value': p_value,
            'Passed': passed,
            'Interpretation': f"Data is {'approximately normal' if passed else 'non-normal'}"
        }
    except Exception as e:
        return {
            'Test': 'DAgostino-Pearson',
            'Error': str(e)
        }

In [39]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 7: MANN-WHITNEY U TEST (NON-PARAMETRIC)
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def mann_whitney_test(real: pd.Series, synthetic: pd.Series) -> Dict:
    """
    Mann-Whitney U test (non-parametric).
    
    Does not assume normality - more robust when data is skewed.
    Tests whether two samples have the same distribution.
    
    H0: Both samples have the same distribution
    H1: Samples have different distributions
    """
    u_stat, p_value = stats.mannwhitneyu(
        real.dropna(), 
        synthetic.dropna(), 
        alternative='two-sided'
    )
    
    passed = p_value > Config.ALPHA
    
    return {
        'Test': 'Mann-Whitney U',
        'Statistic': u_stat,
        'p_value': p_value,
        'Passed': passed,
        'Interpretation': f"Distributions are {'statistically similar' if passed else 'significantly different'}"
    }

In [40]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 8: LEVENE'S TEST FOR VARIANCE EQUALITY
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def levene_test(real: pd.Series, synthetic: pd.Series) -> Dict:
    """
    Levene's test for equality of variances.
    
    Critical for risk modeling where variance (volatility) drives metrics.
    
    H0: Both samples have equal variances
    H1: Samples have different variances
    """
    lev_stat, p_value = stats.levene(real.dropna(), synthetic.dropna())
    
    passed = p_value > Config.ALPHA
    
    return {
        'Test': 'Levene',
        'Statistic': lev_stat,
        'p_value': p_value,
        'Real_Variance': real.var(),
        'Synthetic_Variance': synthetic.var(),
        'Passed': passed,
        'Interpretation': f"Variances are {'equal' if passed else 'significantly different'}"
    }

In [41]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 9: CORRELATION STRUCTURE VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def validate_correlation_structure(
    real_df: pd.DataFrame, 
    synthetic_df: pd.DataFrame,
    var_names: List[str]
) -> Dict:
    """
    Validate that correlation structure is preserved between real and synthetic data.
    
    Critical for:
    - ML model training (relationships must be preserved)
    - Hedging strategies (correlation drives hedge ratios)
    - Portfolio optimization (covariance matrix)
    """
    real_corr = real_df.corr()
    synth_corr = synthetic_df.corr()
    
    # Compute error matrix
    error_matrix = (real_corr - synth_corr).abs()
    
    # Extract pairwise comparisons
    pairs = []
    n = len(var_names)
    for i in range(n):
        for j in range(i+1, n):
            var1, var2 = var_names[i], var_names[j]
            real_c = real_corr.iloc[i, j]
            synth_c = synth_corr.iloc[i, j]
            error = abs(real_c - synth_c)
            error_pct = (error / abs(real_c) * 100) if real_c != 0 else np.inf
            
            # Quality rating
            if error < Config.CORR_ERROR_EXCELLENT:
                quality = "EXCELLENT"
            elif error < Config.CORR_ERROR_GOOD:
                quality = "GOOD"
            elif error < Config.CORR_ERROR_ACCEPTABLE:
                quality = "ACCEPTABLE"
            else:
                quality = "POOR"
            
            pairs.append({
                'Pair': f"{var1}-{var2}",
                'Real_Corr': real_c,
                'Synth_Corr': synth_c,
                'Error': error,
                'Error_Pct': error_pct,
                'Quality': quality
            })
    
    # Summary statistics
    errors = [p['Error'] for p in pairs]
    
    return {
        'Real_Correlation_Matrix': real_corr,
        'Synthetic_Correlation_Matrix': synth_corr,
        'Error_Matrix': error_matrix,
        'Pairwise_Comparison': pairs,
        'Mean_Absolute_Error': np.mean(errors),
        'Max_Absolute_Error': np.max(errors),
        'Std_of_Errors': np.std(errors),
        'Pairs_Excellent': sum(1 for p in pairs if p['Quality'] == 'EXCELLENT'),
        'Pairs_Good': sum(1 for p in pairs if p['Quality'] == 'GOOD'),
        'Pairs_Acceptable': sum(1 for p in pairs if p['Quality'] == 'ACCEPTABLE'),
        'Pairs_Poor': sum(1 for p in pairs if p['Quality'] == 'POOR')
    }

In [42]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# SECTION 10: TAIL RISK ANALYSIS (VaR/CVaR)
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

def analyze_tail_risk(real: pd.Series, synthetic: pd.Series) -> Dict:
    """
    Analyze tail risk preservation for VaR/CVaR modeling.
    
    Compares key percentiles that matter for risk management:
    - 1st percentile (99% VaR)
    - 5th percentile (95% VaR)
    - 95th percentile
    - 99th percentile
    """
    percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    
    comparisons = []
    for p in percentiles:
        real_q = real.quantile(p)
        synth_q = synthetic.quantile(p)
        diff_pct = abs(real_q - synth_q) / abs(real_q) * 100 if real_q != 0 else 0
        
        comparisons.append({
            'Percentile': f"{p*100:.0f}%",
            'Real': real_q,
            'Synthetic': synth_q,
            'Diff_Pct': diff_pct
        })
    
    # Calculate tail coverage ratios
    real_iqr = real.quantile(0.75) - real.quantile(0.25)
    synth_iqr = synthetic.quantile(0.75) - synthetic.quantile(0.25)
    
    real_tail_range = real.quantile(0.99) - real.quantile(0.01)
    synth_tail_range = synthetic.quantile(0.99) - synthetic.quantile(0.01)
    
    return {
        'Percentile_Comparison': comparisons,
        'IQR_Coverage': synth_iqr / real_iqr * 100 if real_iqr != 0 else np.nan,
        'Tail_Range_Coverage': synth_tail_range / real_tail_range * 100 if real_tail_range != 0 else np.nan,
        'VaR_99_Real': real.quantile(0.01),
        'VaR_99_Synthetic': synthetic.quantile(0.01),
        'VaR_95_Real': real.quantile(0.05),
        'VaR_95_Synthetic': synthetic.quantile(0.05)
    }

In [43]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# COMPREHENSIVE VALIDATION SUITE
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

class SyntheticDataValidator:
    """
    Comprehensive validation suite for synthetic data.
    
    Usage:
    ------
    validator = SyntheticDataValidator(real_df, synthetic_df, column_mapping)
    results = validator.run_all_tests()
    validator.generate_report()
    """
    
    def __init__(
        self, 
        real_df: pd.DataFrame, 
        synthetic_df: pd.DataFrame,
        column_mapping: Dict[str, Tuple[str, str]]
    ):
        """
        Initialize validator.
        
        Parameters:
        -----------
        real_df : pd.DataFrame
            Real data
        synthetic_df : pd.DataFrame
            Synthetic data
        column_mapping : dict
            Maps variable names to (real_col, synth_col) tuples
        """
        self.real_df = real_df
        self.synthetic_df = synthetic_df
        self.column_mapping = column_mapping
        self.results = {}
    
    def run_all_tests(self) -> Dict:
        """Run all validation tests."""
        results = {}
        
        for var_name, (real_col, synth_col) in self.column_mapping.items():
            real_series = self.real_df[real_col].dropna()
            synth_series = self.synthetic_df[synth_col].dropna()
            
            results[var_name] = {
                'Descriptive': compare_descriptive_stats(real_series, synth_series, var_name),
                'Distribution_Shape_Real': analyze_distribution_shape(real_series),
                'Distribution_Shape_Synth': analyze_distribution_shape(synth_series),
                'KS_Test': ks_two_sample_test(real_series, synth_series),
                'Mann_Whitney': mann_whitney_test(real_series, synth_series),
                'Levene': levene_test(real_series, synth_series),
                'Jarque_Bera_Real': jarque_bera_test(real_series),
                'Jarque_Bera_Synth': jarque_bera_test(synth_series),
                'Anderson_Darling_Real': anderson_darling_test(real_series),
                'Anderson_Darling_Synth': anderson_darling_test(synth_series),
                'DAgostino_Real': dagostino_pearson_test(real_series),
                'DAgostino_Synth': dagostino_pearson_test(synth_series),
                'Tail_Risk': analyze_tail_risk(real_series, synth_series)
            }
        
        self.results = results
        return results
    
    def get_summary_table(self) -> pd.DataFrame:
        """Generate summary table for presentation."""
        rows = []
        for var_name, tests in self.results.items():
            row = {
                'Variable': var_name,
                'KS_Test': 'PASS' if tests['KS_Test']['Passed'] else 'FAIL',
                'KS_pvalue': tests['KS_Test']['p_value'],
                'Mann_Whitney': 'PASS' if tests['Mann_Whitney']['Passed'] else 'FAIL',
                'MW_pvalue': tests['Mann_Whitney']['p_value'],
                'Levene': 'PASS' if tests['Levene']['Passed'] else 'FAIL',
                'Levene_pvalue': tests['Levene']['p_value'],
                'JB_Synth': 'PASS' if tests['Jarque_Bera_Synth']['Passed'] else 'FAIL',
                'Mean_Error_Pct': tests['Descriptive']['Mean_Error_Pct'],
                'Std_Error_Pct': tests['Descriptive']['Std_Error_Pct']
            }
            rows.append(row)
        
        return pd.DataFrame(rows)
    
    def calculate_overall_score(self) -> float:
        """Calculate overall validation score (0-10)."""
        scores = []
        
        for var_name, tests in self.results.items():
            var_score = 0
            
            # KS test (weight: 2)
            if tests['KS_Test']['Passed']:
                var_score += 2
            elif tests['KS_Test']['p_value'] > 0.01:
                var_score += 1
            
            # Mann-Whitney (weight: 1.5)
            if tests['Mann_Whitney']['Passed']:
                var_score += 1.5
            
            # Levene (weight: 1.5)
            if tests['Levene']['Passed']:
                var_score += 1.5
            
            # Mean error (weight: 2)
            mean_err = tests['Descriptive']['Mean_Error_Pct']
            if mean_err < 1:
                var_score += 2
            elif mean_err < 5:
                var_score += 1.5
            elif mean_err < 10:
                var_score += 1
            
            # Std error (weight: 1.5)
            std_err = tests['Descriptive']['Std_Error_Pct']
            if std_err < 5:
                var_score += 1.5
            elif std_err < 10:
                var_score += 1
            
            # Normalize to 10
            var_score = var_score / 8.5 * 10
            scores.append(var_score)
        
        return np.mean(scores)

In [46]:
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Example usage
    print("=" * 80)
    print("GS QUANTITATIVE STRATEGIES - SYNTHETIC DATA VALIDATION SUITE")
    print("=" * 80)
    
    # Load data
    real_data = pd.read_csv('Data_ML.csv')
    synthetic_data = pd.read_csv('urals_synthetic_data.csv')
    
    # Fix Urals loading parsing
    real_data['Urals loading'] = real_data['Urals loading'].str.replace(',', '').astype(float)
    
    # Define column mapping
    column_mapping = {
        'URALS': ('Urals loading', 'Urals loading'),
        'BRENT': ('LCOc1', 'LCOc1'),
        'MOEX': ('.IMOEX', '.IMOEX'),
        'NWEMURL': ('NWEMURLCRKMc1', 'NWEMURL.CRMc1')
    }
    
    # Initialize validator
    validator = SyntheticDataValidator(real_data, synthetic_data, column_mapping)
    
    # Run all tests
    results = validator.run_all_tests()
    
    # Get summary
    summary = validator.get_summary_table()
    print("\nSUMMARY TABLE:")
    print(summary.to_string(index=False))
    
    # Calculate overall score
    score = validator.calculate_overall_score()
    print(f"\nOVERALL VALIDATION SCORE: {score:.1f}/10")
    
    # Correlation validation
    real_cols = ['Urals loading', 'LCOc1', '.IMOEX', 'NWEMURLCRKMc1']
    synth_cols = ['Urals loading', 'LCOc1', '.IMOEX', 'NWEMURL.CRMc1']
    
    corr_results = validate_correlation_structure(
        real_data[real_cols],
        synthetic_data[synth_cols],
        ['URALS', 'BRENT', 'MOEX', 'NWEMURL']
    )
    
    print("\nCORRELATION PRESERVATION:")
    print(f"Mean Absolute Error: {corr_results['Mean_Absolute_Error']:.4f}")
    print(f"Max Absolute Error: {corr_results['Max_Absolute_Error']:.4f}")
    print(f"Pairs EXCELLENT: {corr_results['Pairs_Excellent']}/{len(corr_results['Pairwise_Comparison'])}")


GS QUANTITATIVE STRATEGIES - SYNTHETIC DATA VALIDATION SUITE

SUMMARY TABLE:
Variable KS_Test  KS_pvalue Mann_Whitney  MW_pvalue Levene  Levene_pvalue JB_Synth  Mean_Error_Pct  Std_Error_Pct
   URALS    PASS   0.466606         PASS   0.777607   PASS       0.127501     PASS        0.501464       8.912962
   BRENT    PASS   0.905819         PASS   0.963315   PASS       0.843935     PASS        0.630584       0.215067
    MOEX    PASS   0.066274         PASS   0.390396   PASS       0.748810     PASS        0.479854       0.596359
 NWEMURL    FAIL   0.048737         PASS   0.755757   PASS       0.072304     PASS        9.322790       0.024015

OVERALL VALIDATION SCORE: 9.3/10

CORRELATION PRESERVATION:
Mean Absolute Error: 0.0470
Max Absolute Error: 0.0726
Pairs EXCELLENT: 2/6
